# Ejercicio 1: una red neuronal frente al ensamblaje

`viajes_features.csv`, con la misma preparación y la misma división de la sesión pasada, importadas de `preparacion_taxis.py`. El gradient boosting de la sesión pasada ya está entrenado en su celda.

Corre todas las celdas antes de tocar nada (**Run All**, en la barra de arriba del notebook): corre de principio a fin sin los `TODO` resueltos. Después resuelve cada `TODO` y vuelve a correr solo la celda de ese modelo y la de la tabla: los demás modelos no se vuelven a entrenar.

In [1]:
import time

import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from preparacion_taxis import dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (2380, 10) Prueba: (595, 10)


## La función que evalúa cada modelo

In [2]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion y mide el error absoluto
    medio en entrenamiento y en prueba. Guarda la fila de la tabla en
    resultados y el pipeline entrenado en entrenados, los dos con su
    nombre: si vuelves a correr la celda de un modelo, se reemplazan, no
    quedan repetidos."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "mae_entrenamiento": round(mae_ent, 4),
        "mae_prueba": round(mae_prueba, 4),
        "brecha": round(mae_prueba - mae_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:28s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   {segundos:6.2f} s")

## El ensamblaje de la sesión pasada

In [3]:
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))
ensamblaje = entrenados["gradient boosting"]

gradient boosting            entrenamiento  1.0705   prueba  1.2754     0.33 s


## Paso 1: una red neuronal

In [4]:
# TODO 1: una red neuronal (MLPRegressor) con una capa oculta de 64
# neuronas, hasta 1000 iteraciones y random_state=42.
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42))
red = entrenados["red neuronal"]

red neuronal                 entrenamiento  1.0882   prueba  1.1695     2.83 s


## Paso 2: cuántos números aprendió la red

In [5]:
# TODO 2: parametros_red, cuantos numeros aprendio la red: la suma de los
# tamanos (.size) de todas las matrices de coefs_ y todos los vectores
# de intercepts_ del paso "modelo".
modelo_red = red.named_steps["modelo"]
parametros_red = sum(m.size for m in modelo_red.coefs_) + sum(v.size for v in modelo_red.intercepts_)
print("Parametros de la red:", parametros_red)

Parametros de la red: 1729


## Paso 3: qué columnas usa más el ensamblaje

In [6]:
# TODO 3: importancias, una serie de pandas con feature_importances_ del
# ensamblaje, indexada por los nombres de columna que produce la
# preparacion (get_feature_names_out), ordenada de mayor a menor.
importancias = pd.Series(
    ensamblaje.named_steps["modelo"].feature_importances_,
    index=ensamblaje.named_steps["preparar"].get_feature_names_out(),
).sort_values(ascending=False)
importancias

numericas__distancia_km                      0.982386
ya_listas__es_tarifa_fija                    0.011997
ya_listas__es_hora_pico                      0.002842
categoricas__zona_origen_JFK Airport         0.001774
categoricas__zona_destino_JFK Airport        0.000366
categoricas__zona_origen_Midtown             0.000279
ya_listas__hora_sin                          0.000080
categoricas__zona_origen_Brooklyn            0.000066
ya_listas__hora_cos                          0.000064
categoricas__zona_destino_Midtown            0.000033
categoricas__zona_origen_Queens              0.000017
categoricas__zona_origen_Harlem              0.000013
categoricas__zona_destino_Harlem             0.000013
categoricas__zona_origen_Upper West Side     0.000013
numericas__pasajeros                         0.000009
categoricas__zona_destino_Upper West Side    0.000009
ya_listas__es_fin_de_semana                  0.000009
categoricas__zona_destino_Brooklyn           0.000007
categoricas__zona_origen_Low

## Tabla de resultados

In [7]:
pd.DataFrame(resultados.values())

,modelo,mae_entrenamiento,mae_prueba,brecha,segundos
0,gradient boosting,1.0705,1.2754,0.2048,0.330
1,red neuronal,1.0882,1.1695,0.0813,2.828
